In [2]:
from equilibrator_api import ComponentContribution, Q_
import matplotlib.pyplot as plt
import numpy as np
import pickle
import os 
# WARNING: THE CODE IS RUNNED WITH THE 
# EQUILIBRATOR_API THEREFORE SHOULD NOT BE RUNNED IF NOT NEEDED
# BECASE IN ORDER TO RUN IT YOU NEED TO INSTALL THE EQUILIBRATOR_API 
# WHICH INSTALLS THE EQUILIBRATOR DATABASE, WHICH IS QUITE HEAVY AND CAN CAUSE PROBLEMS IN THE SYSTEM

In [3]:
R = 8.31446261815324 # J/(mol.K)
T = 37.0 + 273.15 # K, temperature set as ishi mentioned
cc = ComponentContribution()

cc.p_h = Q_(7.0) # pH 
cc.p_mg = Q_(3.0) # Mg2+ concentration
cc.ionic_strength = Q_(0.1, "M") # ionic strength
cc.temperature = Q_(37.0 + 273.15, "K") # temperature set as ishi mentioned

pgi_reaction = cc.parse_reaction_formula(
    'bigg.metabolite:g6p = bigg.metabolite:f6p'
)
dGrm_pgi = cc.physiological_dg_prime(pgi_reaction)

pfk_reaction = cc.parse_reaction_formula(
    'bigg.metabolite:f6p + bigg.metabolite:atp = bigg.metabolite:fdp + bigg.metabolite:adp'
)
dGrm_pfk = cc.physiological_dg_prime(pfk_reaction)

fba_reaction = cc.parse_reaction_formula(
    'bigg.metabolite:fdp = bigg.metabolite:g3p + bigg.metabolite:dhap'
)
dGrm_fba = cc.physiological_dg_prime(fba_reaction)

tpi_reaction = cc.parse_reaction_formula(
    'bigg.metabolite:dhap = bigg.metabolite:g3p'
)
dGrm_tpi = cc.physiological_dg_prime(tpi_reaction)

gap_reaction = cc.parse_reaction_formula(
    'bigg.metabolite:g3p + bigg.metabolite:nad + bigg.metabolite:pi = bigg.metabolite:13dpg + bigg.metabolite:nadh + bigg.metabolite:h'
)
dGrm_gap = cc.physiological_dg_prime(gap_reaction)

pgk_reaction = cc.parse_reaction_formula(
    'bigg.metabolite:13dpg + bigg.metabolite:adp = bigg.metabolite:3pg + bigg.metabolite:atp'
)
dGrm_pgk = cc.physiological_dg_prime(pgk_reaction)


gpm_reaction = cc.parse_reaction_formula(
    'bigg.metabolite:3pg = bigg.metabolite:2pg'
)
dGrm_gpm = cc.physiological_dg_prime(gpm_reaction)

eno_reaction = cc.parse_reaction_formula(
    'bigg.metabolite:2pg = bigg.metabolite:pep + bigg.metabolite:h2o'
)
dGrm_eno = cc.physiological_dg_prime(eno_reaction)

### now the Keq values are computed as Keq = exp(-dGrm / (R * T))

k_eq = {
    # the value.m_as('J/mol') allows to convert the dGrm from kJ/mol to J/mol, which is necessary because R is in J/(mol.K)
    "pgi": np.exp(-dGrm_pgi.value.m_as('J/mol') / (R * T)), # 
    "pfk": np.exp(-dGrm_pfk.value.m_as('J/mol') / (R * T)),
    "fba": np.exp(-dGrm_fba.value.m_as('J/mol') / (R * T)),
    "tpi": np.exp(-dGrm_tpi.value.m_as('J/mol') / (R * T)),
    "gap": np.exp(-dGrm_gap.value.m_as('J/mol') / (R * T)),
    "pgk": np.exp(-dGrm_pgk.value.m_as('J/mol') / (R * T)),
    "gpm": np.exp(-dGrm_gpm.value.m_as('J/mol') / (R * T)),
    "eno": np.exp(-dGrm_eno.value.m_as('J/mol') / (R * T)),
}

with open(os.path.join('Data', 'k_eq_values.pkl'), 'wb') as f:
    pickle.dump(k_eq, f)
    

In [4]:
k_eq

{'pgi': np.float64(0.38101936995267216),
 'pfk': np.float64(341.29799868451624),
 'fba': np.float64(0.1616095034891657),
 'tpi': np.float64(0.1130151288817738),
 'gap': np.float64(0.00026021678575010237),
 'pgk': np.float64(1849.5721766964416),
 'gpm': np.float64(0.18746645814746718),
 'eno': np.float64(4.527341611705358)}